In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

In [2]:
class TaylorDispersionNet(nn.Module):
    def __init__(self, r_max=1.0, z_max=1.0, t_max=1.0):
        super(TaylorDispersionNet, self).__init__()
        
        # Store max domain bounds so we can normalize inputs to [-1, 1]
        self.r_max = r_max
        self.z_max = z_max
        self.t_max = t_max

        # Define layers explicitly (your original structure)
        self.layer1 = nn.Linear(3, 100)
        self.act1 = nn.Tanh()
        self.layer2 = nn.Linear(100, 100)
        self.act2 = nn.Tanh()
        self.layer3 = nn.Linear(100, 100)
        self.act3 = nn.Tanh()
        self.layer4 = nn.Linear(100, 100)
        self.act4 = nn.Tanh()
        self.layer5 = nn.Linear(100, 100)
        self.act5 = nn.Tanh()
        self.layer6 = nn.Linear(100, 100)
        self.act6 = nn.Tanh()
        self.layer7 = nn.Linear(100, 1)

        # Apply Xavier initialization to every linear layer
        for layer in [self.layer1, self.layer2, self.layer3, self.layer4, self.layer5, self.layer6, self.layer7]:
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, r, z, t):
        # Normalize each coordinate from [0, max] to [-1, 1]
        r_norm = 2.0 * (r / self.r_max) - 1.0
        z_norm = 2.0 * (z / self.z_max) - 1.0
        t_norm = 2.0 * (t / self.t_max) - 1.0

        # Concatenate normalized inputs and pass through your layers
        x = torch.cat((r_norm, z_norm, t_norm), dim=1)
        x = self.act1(self.layer1(x))
        x = self.act2(self.layer2(x))
        x = self.act3(self.layer3(x))
        x = self.act4(self.layer4(x))
        x = self.act5(self.layer5(x))
        x = self.act6(self.layer6(x))
        x = self.layer7(x)
        return x


In [ ]:
def pde_residual(model, r, z, t, Pe):
    r.requires_grad = True
    z.requires_grad = True
    t.requires_grad = True

    c = model(r, z, t)

    c_t = torch.autograd.grad(c, t, torch.ones_like(c), create_graph=True)[0]
    c_z   = torch.autograd.grad(c, z, torch.ones_like(c), create_graph=True)[0]
    c_r   = torch.autograd.grad(c, r, torch.ones_like(c), create_graph=True)[0]
    
    c_zz  = torch.autograd.grad(c_z, z, torch.ones_like(c_z), create_graph=True)[0]
    c_rr  = torch.autograd.grad(c_r, r, torch.ones_like(c_r), create_graph=True)[0]

    eps = 1e-7
    radial_diffusion = torch.where(
        r > eps,
        c_rr + (1.0 / r) * c_r,
        2.0 * c_rr
    )

    u_z = 2.0 * (1.0 - r ** 2)
    residual = c_t + u_z * c_z - (1.0 / Pe) * (c_zz + radial_diffusion)
    
    return residual


In [ ]:
# 1. INITIAL CONDITION at t = 0 (Dimensionless step function)
# Solute completely fills the tube radius (r from 0 to 1) from z = 0 up to z_step.
def initial_condition(r, z, t, C0=1.0, z_step=0.8):
    # Enforces a flat block of concentration C0 over a fraction of the dimensionless length.
    # Adjust z_step based on your dimensionless axial length (e.g., if z max is 10, z_step=1.0 is 10%).
    c_init = torch.where(z <= z_step, torch.tensor(C0, device=z.device), torch.tensor(0.0, device=z.device))
    return c_init

# 2. INLET BOUNDARY CONDITION at z = 0 (t > 0)
# Clean fluid enters the domain behind the moving slug.
def inlet_boundary_condition(r, z, t):
    return torch.zeros_like(z)  # c(r, 0, t) = 0

# 3. OUTLET BOUNDARY CONDITION at z = z_max
# Enforcing a zero-gradient boundary condition prevents artificial back-reflection.
def outlet_boundary_condition_residual(model, r, z, t):
    z.requires_grad = True
    # Ensure variables match how you pass them to your model
    c = model(r, z, t)
    c_z = torch.autograd.grad(c, z, torch.ones_like(c), create_graph=True)[0]
    return c_z  # Target loss drives c_z -> 0

# 4. IMPERMEABLE WALL CONDITION at r = 1.0 (Dimensionless Pipe Wall)
# Because it is dimensionless, the wall is strictly at r = 1.0.
def wall_boundary_condition_residual(model, r, z, t):
    r.requires_grad = True
    c = model(r, z, t)
    c_r = torch.autograd.grad(c, r, torch.ones_like(c), create_graph=True)[0]
    return c_r  # Target loss drives c_r -> 0

# 5. CENTERLINE SYMMETRY CONDITION at r = 0.0
# Smooth profile across the axis.
def symmetry_boundary_condition_residual(model, r, z, t):
    r.requires_grad = True
    c = model(r, z, t)
    c_r = torch.autograd.grad(c, r, torch.ones_like(c), create_graph=True)[0]
    return c_r  # Target loss drives c_r -> 0


In [ ]:
# creating a function to generate initial condition loss
def initial_condition_loss(model, r, z, t, C0=1.0, z_step=1.0):
    c_pred = model(r, z, t)
    c_init = initial_condition(r, z, t, C0=C0, z_step=z_step)
    loss_ic = torch.mean((c_pred - c_init) ** 2)
    return loss_ic

def inlet_boundary_condition_loss(model, r, z, t):
    c_pred = model(r, z, t)
    c_inlet = inlet_boundary_condition(r, z, t)
    loss_inlet = torch.mean((c_pred - c_inlet) ** 2)
    return loss_inlet

def outlet_boundary_condition_loss(model, r, z, t):
    c_z_pred = outlet_boundary_condition_residual(model, r, z, t)
    loss_outlet = torch.mean(c_z_pred ** 2)
    return loss_outlet

def wall_boundary_condition_loss(model, r, z, t):
    c_r_pred = wall_boundary_condition_residual(model, r, z, t)
    loss_wall = torch.mean(c_r_pred ** 2)
    return loss_wall

def symmetry_boundary_condition_loss(model, r, z, t):
    c_r_pred = symmetry_boundary_condition_residual(model, r, z, t)
    loss_symmetry = torch.mean(c_r_pred **2)
    return loss_symmetry

In [ ]:
# Define domain bounds
R_max = 1.0
Z_max = 22.0
T_max = 16.0

# 1. INTERIOR COLLOCATION POINTS (For pde_residual)
# Instead of a heavy meshgrid, PINNs train best using randomly sampled points (Monte Carlo)
N_pde = 20000  # 20k points is plenty for a 2D axisymmetric transient problem
r_pde = torch.rand(N_pde, 1) * R_max
z_pde = torch.rand(N_pde, 1) * Z_max
t_pde = torch.rand(N_pde, 1) * T_max

# 2. INITIAL CONDITION POINTS (t = 0)
N_ic = 2000
r_ic = torch.rand(N_ic, 1) * R_max
z_ic = torch.rand(N_ic, 1) * Z_max
t_ic = torch.zeros(N_ic, 1)  # Strictly t = 0

# 3. INLET BOUNDARY POINTS (z = 0)
N_inlet = 2000
r_inlet = torch.rand(N_inlet, 1) * R_max
z_inlet = torch.zeros(N_inlet, 1)  # Strictly z = 0
t_inlet = torch.rand(N_inlet, 1) * T_max

# 4. OUTLET BOUNDARY POINTS (z = L)
N_outlet = 2000
r_outlet = torch.rand(N_outlet, 1) * R_max
z_outlet = torch.ones(N_outlet, 1) * Z_max  # Strictly z = Z_max
t_outlet = torch.rand(N_outlet, 1) * T_max

# 5. WALL BOUNDARY POINTS (r = R)
N_wall = 2000
r_wall = torch.ones(N_wall, 1) * R_max  # Strictly r = R_max
z_wall = torch.rand(N_wall, 1) * Z_max
t_wall = torch.rand(N_wall, 1) * T_max

# 6. CENTERLINE SYMMETRY POINTS (r = 0)
N_sym = 2000
r_sym = torch.zeros(N_sym, 1)  # Strictly r = 0
z_sym = torch.rand(N_sym, 1) * Z_max
t_sym = torch.rand(N_sym, 1) * T_max

model = TaylorDispersionNet()

In [ ]:
# Ensure everything runs on GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#model = TaylorDispersionNet().to(device)
Pe = 30.0
z_step = 1.0

model = TaylorDispersionNet(r_max=R_max, z_max=Z_max, t_max=T_max).to(device)


# Move all your generated datasets to the active device
# (Assuming r_pde, z_pde, t_pde, etc. were created as shown previously)
r_pde, z_pde, t_pde = r_pde.to(device), z_pde.to(device), t_pde.to(device)
r_ic, z_ic, t_ic = r_ic.to(device), z_ic.to(device), t_ic.to(device)
r_inlet, z_inlet, t_inlet = r_inlet.to(device), z_inlet.to(device), t_inlet.to(device)
r_outlet, z_outlet, t_outlet = r_outlet.to(device), z_outlet.to(device), t_outlet.to(device)
r_wall, z_wall, t_wall = r_wall.to(device), z_wall.to(device), t_wall.to(device)
r_sym, z_sym, t_sym = r_sym.to(device), z_sym.to(device), t_sym.to(device)

# Dimensionless flow parameter
Pe = 100  # Example Peclet number

# Define a helper function to compute total weighted loss
def compute_total_loss():
    # 1. PDE Residual Loss (Note: ensure your pde_residual accepts r, z, t separately!)
    res_pde = pde_residual(model, r_pde, z_pde, t_pde, Pe)
    loss_pde = torch.mean(res_pde ** 2)

    # 2. Initial and Boundary Condition Losses
    loss_ic = initial_condition_loss(model, r_ic, z_ic, t_ic, C0=1.0, z_step=0.1)
    loss_inlet = inlet_boundary_condition_loss(model, r_inlet, z_inlet, t_inlet)
    loss_outlet = outlet_boundary_condition_loss(model, r_outlet, z_outlet, t_outlet)
    loss_wall = wall_boundary_condition_loss(model, r_wall, z_wall, t_wall)
    loss_sym = symmetry_boundary_condition_loss(model, r_sym, z_sym, t_sym)

    # Weighting: Initial and Inlet conditions often need higher weights (e.g., 10.0) 
    # so the network doesn't cheat by predicting c = 0 everywhere!
    w_ic = 10.0
    w_bc = 1.0
    w_pde = 1.0

    total_loss = (w_pde * loss_pde) + (w_ic * loss_ic) + w_bc * (loss_inlet + loss_outlet + loss_wall + loss_sym)
    return total_loss, loss_pde, loss_ic

# ==========================================
# PHASE 1: ADAM OPTIMIZER (Global Search)
# ==========================================
optimizer_adam = torch.optim.Adam(model.parameters(), lr=0.001)
adam_epochs = 5000

print("--- Starting Phase 1: Adam Optimization ---")
for epoch in range(adam_epochs):
    optimizer_adam.zero_grad()
    
    total_loss, loss_pde, loss_ic = compute_total_loss()
    
    total_loss.backward()
    optimizer_adam.step()
    
    if (epoch + 1) % 500 == 0:
        print(f"Adam Epoch [{epoch+1}/{adam_epochs}] | Total Loss: {total_loss.item():.6f} | PDE: {loss_pde.item():.6f} | IC: {loss_ic.item():.6f}")

# ==========================================
# PHASE 2: L-BFGS OPTIMIZER (High-Precision Fine Tuning)
# ==========================================
optimizer_lbfgs = torch.optim.LBFGS(
    model.parameters(), 
    lr=1.0, 
    max_iter=2000, 
    max_eval=2500, 
    history_size=50,
    tolerance_grad=1e-7, 
    tolerance_change=1e-9,
    line_search_fn="strong_wolfe"
)

# L-BFGS requires a closure function that re-evaluates the model and returns the loss
lbfgs_step = 0
def closure():
    global lbfgs_step
    optimizer_lbfgs.zero_grad()
    total_loss, loss_pde, loss_ic = compute_total_loss()
    total_loss.backward()
    
    lbfgs_step += 1
    if lbfgs_step % 250 == 0:
        print(f"L-BFGS Step [{lbfgs_step}] | Total Loss: {total_loss.item():.6f} | PDE: {loss_pde.item():.6f} | IC: {loss_ic.item():.6f}")
        
    return total_loss

print("\n--- Starting Phase 2: L-BFGS Optimization ---")
optimizer_lbfgs.step(closure)
print("Training Complete!")

--- Starting Phase 1: Adam Optimization ---
Adam Epoch [500/5000] | Total Loss: 0.172846 | PDE: 0.063698 | IC: 0.009218
Adam Epoch [1000/5000] | Total Loss: 0.091034 | PDE: 0.009291 | IC: 0.007746
Adam Epoch [1500/5000] | Total Loss: 0.034157 | PDE: 0.007254 | IC: 0.001987
Adam Epoch [2000/5000] | Total Loss: 0.025944 | PDE: 0.004161 | IC: 0.001688
Adam Epoch [2500/5000] | Total Loss: 0.053086 | PDE: 0.009375 | IC: 0.003802
Adam Epoch [3000/5000] | Total Loss: 0.027894 | PDE: 0.006217 | IC: 0.001731
Adam Epoch [3500/5000] | Total Loss: 0.010153 | PDE: 0.000625 | IC: 0.000709
Adam Epoch [4000/5000] | Total Loss: 0.110852 | PDE: 0.014348 | IC: 0.009257
Adam Epoch [4500/5000] | Total Loss: 0.044200 | PDE: 0.011683 | IC: 0.002845
Adam Epoch [5000/5000] | Total Loss: 0.027848 | PDE: 0.005002 | IC: 0.001984

--- Starting Phase 2: L-BFGS Optimization ---
L-BFGS Step [250] | Total Loss: 0.002624 | PDE: 0.000324 | IC: 0.000180
L-BFGS Step [500] | Total Loss: 0.000028 | PDE: 0.000010 | IC: 0.000

In [ ]:
import torch
import numpy as np
from scipy.special import erf
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import LinearSegmentedColormap

# ==========================================
# 1. SIMULATION PARAMETERS (Match your PINN)
# ==========================================
Pe = 100.0        # Peclet number used in training
C0 = 1.0          # Initial slug concentration
z_step = 0.1      # Initial slug length (0 to z_step)
Z_max = 1.0       # Axial domain length
T_max = 1.0       # Total simulation time

# Effective Aris-Taylor parameters (dimensionless)
u_avg = Pe / 2.0
D_eff = 1.0 + (Pe ** 2) / 192.0

# ==========================================
# 2. CUSTOM THERMAL CYAN -> HOT PINK PALETTE
# ==========================================
BG_DARK = '#080B12'       # Deep obsidian figure background
PANEL_BG = '#0D121D'      # Slightly lighter dark slate for axes
TEXT_COLOR = '#E6EDF3'    # Crisp off-white for labels
MUTED_COLOR = '#8B949E'   # Subtle gray for grids/spines
CYAN_ACCENT = '#00F5D4'   # Electric cyan (PINN curve)
PINK_ACCENT = '#FF007F'   # Hot pink / magenta (Analytical limit)

# Custom perceptually smooth dark-thermal colormap: Obsidian -> Deep Blue -> Cyan -> Violet -> Hot Pink
thermal_colors = [
    '#080B12',  # 0.00: Background zero-concentration
    '#0B2545',  # 0.20: Deep midnight blue
    '#00B4D8',  # 0.45: Bright cerulean
    '#00F5D4',  # 0.65: Electric cyan
    '#9B5DE5',  # 0.82: Neon violet transition
    '#FF007F',  # 1.00: Peak hot pink
]
cmap_thermal = LinearSegmentedColormap.from_list('cyan_hotpink_thermal', thermal_colors, N=256)

# ==========================================
# 3. EVALUATE PINN ON DENSE GRID
# ==========================================
model.eval()
device = next(model.parameters()).device

nr = 160   # Radial resolution
nz = 420   # Axial resolution
nt = 120   # Total animation frames

r_vals = np.linspace(0, 1.0, nr)
z_vals = np.linspace(0, Z_max, nz)
#t_vals = np.linspace(0, T_max, nt)
T_vis_max = 0.025  # Zoom into the exact window before the slug exits Z_max = 1.0
t_vals = np.linspace(0, T_vis_max, nt)

Z_grid, R_grid = np.meshgrid(z_vals, r_vals)

# Mirror coordinates across centerline (r = 0) for full-pipe visualization
R_full = np.concatenate([-np.flipud(R_grid[1:]), R_grid], axis=0)
Z_full = np.concatenate([np.flipud(Z_grid[1:]), Z_grid], axis=0)

predictions_2d = []
area_avg_pinn = []
area_avg_analytical = []
total_mass_pinn = []

with torch.no_grad():
    r_flat = torch.tensor(R_grid.flatten(), dtype=torch.float32, device=device).view(-1, 1)
    z_flat = torch.tensor(Z_grid.flatten(), dtype=torch.float32, device=device).view(-1, 1)
    
    for t_step in t_vals:
        t_flat = torch.full_like(r_flat, t_step)
        c_pred = model(r_flat, z_flat, t_flat).cpu().numpy().reshape(nr, nz)
        
        # Mirror for 2D contour plot
        c_full = np.concatenate([np.flipud(c_pred[1:]), c_pred], axis=0)
        predictions_2d.append(c_full)
        
        # 1D Cross-Sectional Area Average: 2 * integral_0^1 c(r,z,t) * r dr
        integrand = c_pred * r_vals[:, None]
        c_bar = 2.0 * np.trapz(integrand, r_vals, axis=0)
        area_avg_pinn.append(c_bar)
        
        # Solute mass conservation check
        mass = np.trapz(c_bar, z_vals)
        total_mass_pinn.append(mass)
        
        # Exact Analytical Aris-Taylor 1D Solution
        #if t_step == 0:
        #    c_ana = np.where(z_vals <= z_step, C0, 0.0)
        #else:
        #    denom = np.sqrt(4.0 * D_eff * t_step)
        #    term1 = erf((z_vals - u_avg * t_step) / denom)
        #    term2 = erf((z_vals - z_step - u_avg * t_step) / denom)
        #    c_ana = 0.5 * C0 * (term1 - term2)
        #area_avg_analytical.append(c_ana)
        
        Pe = 30.0
        Z_max = 22.0
        T_max = 16.0
        z_step = 1.0

        u_avg_adv = 1.0
        D_eff_adv = (1.0 / Pe) + (Pe / 48.0)

        # Inside your frame loop over t_vals:
        if t_step == 0:
            c_ana = np.where(z_vals <= z_step, C0, 0.0)
        else:
            denom = np.sqrt(4.0 * D_eff_adv * t_step)
            term1 = erf((z_vals - u_avg_adv * t_step) / denom)
            term2 = erf((z_vals - z_step - u_avg_adv * t_step) / denom)
            c_ana = 0.5 * C0 * (term1 - term2)


# ==========================================
# 4. JOURNAL-STYLE FIGURE CONFIGURATION
# ==========================================
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'mathtext.fontset': 'dejavusans',
    'axes.edgecolor': MUTED_COLOR,
    'axes.linewidth': 0.9,
    'xtick.color': TEXT_COLOR,
    'ytick.color': TEXT_COLOR,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
})

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(10.5, 6.6), dpi=200,
    facecolor=BG_DARK,
    gridspec_kw={'height_ratios': [1.05, 1.25], 'hspace': 0.36}
)

levels = np.linspace(0.0, C0, 80)
initial_mass = C0 * z_step

# Initial colorbar setup (created once outside the loop)
contour0 = ax1.contourf(Z_full, R_full, predictions_2d[0], levels=levels, cmap=cmap_thermal, extend='both')
cbar = fig.colorbar(contour0, ax=ax1, pad=0.015, aspect=16)
cbar.set_label('Normalized Concentration ($C / C_0$)', color=TEXT_COLOR, fontsize=10, fontweight='bold', labelpad=8)
cbar.outline.set_edgecolor(MUTED_COLOR)
cbar.outline.set_linewidth(0.8)
cbar.ax.yaxis.set_tick_params(color=TEXT_COLOR, labelcolor=TEXT_COLOR, labelsize=9)
cbar.set_ticks([0.0, 0.25, 0.5, 0.75, 1.0])

# ==========================================
# 5. ANIMATION UPDATE LOOP
# ==========================================
def update(frame):
    t_curr = t_vals[frame]
    mass_err = abs(total_mass_pinn[frame] - initial_mass) / initial_mass * 100.0
    
    # --- TOP PANEL: 2D Axisymmetric Pipe Contour ---
    ax1.clear()
    ax1.set_facecolor(PANEL_BG)
    ax1.contourf(Z_full, R_full, predictions_2d[frame], levels=levels, cmap=cmap_thermal, extend='both')
    
    # Subtle iso-concentration contour lines for journal precision
    ax1.contour(Z_full, R_full, predictions_2d[frame], levels=[0.2, 0.5, 0.8], colors='#FFFFFF', linewidths=0.45, alpha=0.35)
    
    # Centerline & Pipe Walls
    ax1.axhline(0, color='#FFFFFF', linestyle=':', linewidth=0.9, alpha=0.45)
    ax1.axhline(1.0, color=CYAN_ACCENT, linewidth=1.2, alpha=0.8)
    ax1.axhline(-1.0, color=CYAN_ACCENT, linewidth=1.2, alpha=0.8)
    
    ax1.set_ylabel('Radial ($r / R$)', color=TEXT_COLOR, fontsize=10.5, fontweight='bold')
    ax1.set_xlim(0, Z_max)
    ax1.set_ylim(-1.0, 1.0)
    ax1.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    ax1.tick_params(top=True, right=True, labelsize=9.5)
    
    ax1.set_title(
        f'2D Axisymmetric Taylor Dispersion (Pe = {Pe:.0f})   |   $t^* = {t_curr:.3f}$   |   Mass Error: {mass_err:.2f}%',
        color=TEXT_COLOR, fontsize=11.5, fontweight='bold', pad=10
    )
    
    # --- BOTTOM PANEL: 1D Area-Average vs. Analytical Aris-Taylor ---
    ax2.clear()
    ax2.set_facecolor(PANEL_BG)
    
    c_pinn = area_avg_pinn[frame]
    c_ana = area_avg_analytical[frame]
    
    # Translucent cyan under-curve glow + crisp primary curve
    ax2.fill_between(z_vals, 0, c_pinn, color=CYAN_ACCENT, alpha=0.14)
    ax2.plot(z_vals, c_pinn, color=CYAN_ACCENT, lw=2.4, label='PINN Cross-Sectional Average $\\bar{c}(z,t)$')
    
    # Hot pink dashed analytical curve
    ax2.plot(z_vals, c_ana, color=PINK_ACCENT, lw=2.1, linestyle='--', label='Aris-Taylor Analytical ($D_{\\mathrm{eff}} = 1 + \\mathrm{Pe}^2/192$)')
    
    # Regime badge
    regime_text = "PRE-ASYMPTOTIC (ADVECTION STRETCHED)" if t_curr < 0.25 else "TAYLOR REGIME (GAUSSIAN DISPERSION)"
    badge_color = PINK_ACCENT if t_curr < 0.25 else CYAN_ACCENT
    ax2.text(
        0.985, 0.91, regime_text, transform=ax2.transAxes, ha='right', va='top',
        color=badge_color, fontsize=8.5, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.35', facecolor='#131B2B', edgecolor=badge_color, linewidth=0.9)
    )
    
    ax2.set_xlabel('Dimensionless Axial Position ($z / R$)', color=TEXT_COLOR, fontsize=10.5, fontweight='bold')
    ax2.set_ylabel('Area-Avg Conc. ($\\bar{c}$)', color=TEXT_COLOR, fontsize=10.5, fontweight='bold')
    ax2.set_xlim(0, Z_max)
    ax2.set_ylim(-0.04, C0 * 1.12)
    ax2.tick_params(top=True, right=True, labelsize=9.5)
    
    # Clean journal grid & legend
    ax2.grid(True, linestyle='--', linewidth=0.6, color=MUTED_COLOR, alpha=0.25)
    legend = ax2.legend(
        loc='upper right', bbox_to_anchor=(0.985, 0.77),
        frameon=True, facecolor='#131B2B', edgecolor=MUTED_COLOR, fontsize=9
    )
    for text in legend.get_texts():
        text.set_color(TEXT_COLOR)

# ==========================================
# 6. EXPORT HIGH-RES MP4 FOR LINKEDIN
# ==========================================
ani = animation.FuncAnimation(fig, update, frames=nt, interval=45, blit=False)

try:
    ani.save('taylor_dispersion_dark_journalt2.mp4', writer='ffmpeg', fps=24, dpi=200)
    print("Saved taylor_dispersion_dark_journalt2.mp4 successfully!")
except Exception as e:
    print(f"FFmpeg not found ({e}), falling back to GIF...")
    ani.save('taylor_dispersion_dark_journalt2.gif', writer='pillow', fps=20)
    print("Saved taylor_dispersion_dark_journalt2.gif successfully!")

plt.close()



MovieWriter ffmpeg unavailable; using Pillow instead.


FFmpeg not found (unknown file extension: .mp4), falling back to GIF...
Saved taylor_dispersion_dark_journalt2.gif successfully!
